# Advanced Kubeflow Pipelines: Basic Triggers

### What you will learn
- How to configure time-based pipeline triggers using KFP schedules.
- How to set up event-based triggers (webhook, data arrival).
- Using Kubernetes CronJobs to orchestrate KFP pipeline runs.

### Why this matters in MLOps
Manual pipeline execution does not scale. Automating triggers — whether on a schedule or in response to events — is what transforms a one-time training script into a production MLOps system. Triggers ensure models are retrained, evaluated, and deployed without human intervention.

### You're done when...
- A time-based pipeline schedule is defined and submitted to KFP.
- An event-based trigger configuration is created.
- You have completed the fill-in-the-blank exercises.

## The MLOps Perspective
Pipeline triggers are the "when" of your ML workflow. In production, models must be retrained periodically as new data arrives (time-based) or reactively when significant events occur (event-based). KFP provides first-class support for both patterns through its Schedule and Trigger APIs, enabling fully automated retraining pipelines that operate with minimal manual oversight.

## Setup and Imports
Ensure the required libraries are installed:

In [ ]:
# Install requirements
!pip install -r requirements.txt

In [ ]:
# Import required libraries
import kfp
from kfp import dsl
from kfp import client
from kfp.dsl import component
import json
from datetime import datetime

## Time-Based Pipeline Triggers
Time-based triggers run pipelines on a recurring schedule using cron expressions. This is the most common pattern for retraining models on a regular cadence (daily, weekly, monthly).

In [ ]:
# Create a simple pipeline component
@component(
    base_image="python:3.11",
    packages_to_install=["pandas"]
)
def train_model(data_path: str, learning_rate: float) -> str:
    """Simulate a model training step."""
    print(f"Training model on data from {data_path}")
    print(f"Using learning rate: {learning_rate}")
    model_uri = f"models/prod_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    print(f"Model saved to: {model_uri}")
    return model_uri

In [ ]:
# Define a pipeline with time-based schedule configuration
@dsl.pipeline(
    name="scheduled-training-pipeline",
    description="A pipeline that trains a model on a recurring schedule"
)
def scheduled_training_pipeline(data_path: str = "/data/latest", learning_rate: float = 0.01):
    train_task = train_model(data_path=data_path, learning_rate=learning_rate)
    train_task.set_caching_options(enable_caching=False)

# Create the recurring run configuration
from kfp.dsl import Schedule

schedule = Schedule(
    cron_expression="0 6 * * *",  # Run daily at 6 AM
    pipeline=scheduled_training_pipeline,
    experiment_name="daily-retraining",
    enable_caching=False
)

## Event-Based Triggers
Event-based triggers respond to external signals such as webhook calls, new data arriving in a bucket, or a model performance alert. KFP integrates with event-driven architectures via its Trigger API.

In [ ]:
# Define a trigger configuration for event-based pipeline execution
@component(
    base_image="python:3.11",
    packages_to_install=["requests"]
)
def check_data_arrival(source_url: str, expected_pattern: str) -> bool:
    """Check if new data matching the expected pattern has arrived."""
    import requests
    try:
        response = requests.head(source_url)
        if response.status_code == 200:
            print(f"Data available at {source_url}")
            return True
    except Exception as e:
        print(f"Data not available: {e}")
    return False


@component(
    base_image="python:3.11"
)
def process_new_data(data_path: str) -> dict:
    """Process newly arrived data."""
    import hashlib
    fingerprint = hashlib.md5(data_path.encode()).hexdigest()
    result = {
        "data_path": data_path,
        "fingerprint": fingerprint,
        "processed": True
    }
    print(f"Processed data: {result}")
    return result


@dsl.pipeline(
    name="event-driven-pipeline",
    description="Pipeline triggered by data arrival events"
)
def event_driven_pipeline(source_url: str = "https://data.example.com/latest", pattern: str = "*.csv"):
    check_task = check_data_arrival(source_url=source_url, expected_pattern=pattern)
    with dsl.Condition(check_task.output == True):
        process_new_data(data_path=source_url)

## Kubernetes CronJob Integration
KFP pipelines can also be triggered externally using Kubernetes CronJobs. This approach works well in environments where you want to decouple the trigger mechanism from KFP itself.

In [ ]:
# Generate a recurring run YAML suitable for Kubernetes CronJob execution
from kubernetes import client as k8s_client
from kubernetes import config

# Create a KFP client
kfp_endpoint = "http://ml-pipeline.kubeflow.svc.cluster.local:8888"
kfp_client = kfp.Client(host=kfp_endpoint)

# Define recurring run configuration
experiment_name = "cronjob-triggered-pipeline"
run_name = f"scheduled-run-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

try:
    experiment = kfp_client.create_experiment(
        name=experiment_name,
        description="Experiment triggered by Kubernetes CronJob"
    )
    print(f"Experiment created: {experiment.name}")
except Exception as e:
    print(f"Experiment may already exist: {e}")

## Fill-in-the-Blank Exercises
Complete the following exercises to reinforce your understanding.

In [ ]:
# Exercise 1: Complete the cron expression for a weekly trigger that runs
# every Monday at 9:00 AM
weekly_cron = "___"
print(f"Weekly schedule: {weekly_cron}")
# Hint: Cron format is minute hour day-of-month month day-of-week

In [ ]:
# Exercise 2: Complete the pipeline trigger configuration below
# so that the pipeline runs when a webhook event is received
from kfp.dsl import ___  # Import the trigger class

webhook_trigger = ___(
    pipeline=scheduled_training_pipeline,
    trigger_type="webhook",
    event_filter={"source": "data-update-topic"}
)
print("Webhook trigger configured")
# Hint: The class name is related to triggering pipelines conditionally

## Summary
In this notebook, you learned:
- Configuring time-based pipeline triggers with cron schedules
- Setting up event-driven pipeline execution
- Integrating KFP with Kubernetes CronJobs
- These patterns form the foundation of automated MLOps workflows